# SCADA processing with aligned alarm codes

This notebook uses the project's existing preparation code to attach each in-scope alarm code to its matching 10-minute SCADA row. SCADA `TimeStamp` marks the **end** of the interval. An alarm starting between 1:10 and 1:20 is assigned to the 1:20 row, representing the interval from 1:10 to 1:20. An alarm exactly at an interval boundary stays on that boundary.

Alarm timestamps are aligned by `build_alarm_table.py` / `wt_predmaint.data.alarms.build_alarm_events` using ceiling to the next 10-minute timestamp; this notebook reuses those prepared event timestamps rather than reimplementing alignment. Only the in-scope alarm codes configured in `configs/protocol.yaml` are included. Multiple codes on the same turbine and interval are combined into a comma-separated `Alarmcode` value, while `alarm_count` retains the number of event records. SCADA intervals without an alarm have a missing `Alarmcode`, `alarm_count` 0, and `alarm_occurred` false.

By default, the notebook uses existing prepared files. Set `RUN_PREPARATION = True` below to run SCADA conversion, alarm-table creation, and SCADA cleaning first. The raw files under `data/raw/` must be available for that step.

In [1]:
from pathlib import Path
import sys
import subprocess

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/protocol.yaml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the repository.")
sys.path.insert(0, str(ROOT / "src"))

YEAR = 2018
STATION = 2304510  # Turbine shown in the preview; processing covers all turbines.
RUN_PREPARATION = False
INTERIM = ROOT / "data/interim"
CLEAN_DIR = INTERIM / "clean"

if RUN_PREPARATION:
    for script in ["build_scada_parquet.py", "build_alarm_table.py", "build_clean_data.py"]:
        print(f"Running {script}...", flush=True)
        subprocess.run([sys.executable, str(ROOT / "scripts" / script)], cwd=ROOT, check=True)
else:
    print("Using existing prepared data. Set RUN_PREPARATION=True to rebuild it.")

Using existing prepared data. Set RUN_PREPARATION=True to rebuild it.


## Load prepared SCADA and alarm events

The clean SCADA file is scoped by the interval's **start year**, whereas its `TimeStamp` is the interval end. Apply the same start-year interpretation to alarm timestamps when selecting events for the chosen year. This also handles the midnight boundary consistently.

In [2]:
scada_path = CLEAN_DIR / f"scada_{YEAR}.parquet"
alarm_path = INTERIM / "alarm_events.parquet"
if not scada_path.exists():
    raise FileNotFoundError(f"Prepared clean SCADA file not found: {scada_path}")
if not alarm_path.exists():
    raise FileNotFoundError(f"Prepared alarm-event file not found: {alarm_path}")

scada = pd.read_parquet(scada_path)
alarms = pd.read_parquet(alarm_path)
required_scada = {"StationId", "TimeStamp"}
required_alarms = {"StationId", "Alarmcode", "TimeStamp"}
if required_scada - set(scada.columns):
    raise ValueError(f"Clean SCADA is missing columns: {sorted(required_scada - set(scada.columns))}")
if required_alarms - set(alarms.columns):
    raise ValueError(f"Alarm events are missing columns: {sorted(required_alarms - set(alarms.columns))}")

scada["TimeStamp"] = pd.to_datetime(scada["TimeStamp"])
alarms["TimeStamp"] = pd.to_datetime(alarms["TimeStamp"])
if scada.duplicated(["StationId", "TimeStamp"]).any():
    raise ValueError("Clean SCADA must have unique turbine/timestamp keys before joining alarms.")

alarm_start = alarms["TimeStamp"] - pd.Timedelta(minutes=10)
alarms_year = alarms.loc[alarm_start.dt.year.eq(YEAR)].copy()
print(f"{YEAR}: {len(scada):,} clean SCADA intervals; {len(alarms_year):,} in-scope alarm events")

2018: 1,094,731 clean SCADA intervals; 633,754 in-scope alarm events


## Attach codes without duplicating SCADA rows

An interval can contain more than one alarm event, so first aggregate alarm codes by turbine and interval. A left join keeps every clean SCADA row, and `validate="one_to_one"` guards against accidentally multiplying rows.

In [3]:
if "Alarmcode" in scada.columns:
    raise ValueError("Clean SCADA already contains an Alarmcode column; inspect it before joining.")

alarm_by_interval = (
    alarms_year.groupby(["StationId", "TimeStamp"], as_index=False)
    .agg(
        Alarmcode=("Alarmcode", lambda codes: ", ".join(str(code) for code in sorted(set(codes)))),
        alarm_count=("Alarmcode", "size"),
    )
)

scada_with_alarms = scada.merge(
    alarm_by_interval,
    on=["StationId", "TimeStamp"],
    how="left",
    validate="one_to_one",
)
scada_with_alarms["alarm_occurred"] = scada_with_alarms["alarm_count"].notna()
scada_with_alarms["alarm_count"] = scada_with_alarms["alarm_count"].fillna(0).astype("int64")

if len(scada_with_alarms) != len(scada):
    raise AssertionError("Alarm join unexpectedly changed the number of SCADA rows.")

matched_events = int(scada_with_alarms["alarm_count"].sum())
unmatched_events = len(alarms_year) - matched_events
print(f"SCADA rows before/after join: {len(scada):,} / {len(scada_with_alarms):,}")
print(f"Alarm events matched to a clean SCADA interval: {matched_events:,}")
print(f"Alarm events without a matching clean SCADA interval: {unmatched_events:,}")

SCADA rows before/after join: 1,094,731 / 1,094,731
Alarm events matched to a clean SCADA interval: 633,147
Alarm events without a matching clean SCADA interval: 607


## Inspect a turbine's rows

The first table shows intervals where at least one configured alarm matched. The second shows a short continuous SCADA slice around the first alarm, including the preceding/following rows so the placement is easy to inspect. `TimeStamp` is the interval end.

In [4]:
station_rows = scada_with_alarms.loc[scada_with_alarms["StationId"].eq(STATION)].sort_values("TimeStamp")
alarm_rows = station_rows.loc[station_rows["alarm_occurred"]]
print(f"Station {STATION}: {len(alarm_rows):,} intervals have one or more matching alarms in {YEAR}.")
display(alarm_rows[["StationId", "TimeStamp", "Alarmcode", "alarm_count", "alarm_occurred"]].head(20))

if not alarm_rows.empty:
    first_alarm_time = alarm_rows["TimeStamp"].iloc[0]
    preview = station_rows.loc[
        station_rows["TimeStamp"].between(
            first_alarm_time - pd.Timedelta(minutes=20),
            first_alarm_time + pd.Timedelta(minutes=20),
        ),
        ["StationId", "TimeStamp", "Alarmcode", "alarm_count", "alarm_occurred"],
    ]
    display(preview)
else:
    print("No matching in-scope alarms for this turbine/year; change STATION or YEAR to inspect another slice.")

Station 2304510: 6,970 intervals have one or more matching alarms in 2018.


,StationId,TimeStamp,Alarmcode,alarm_count,alarm_occurred
210,2304510,2018-01-01 01:50:00,"20, 25",6,True
231,2304510,2018-01-01 02:00:00,"20, 25",2,True
293,2304510,2018-01-01 02:40:00,"20, 25",2,True
303,2304510,2018-01-01 02:50:00,"20, 25",5,True
313,2304510,2018-01-01 03:00:00,20,1,True
323,2304510,2018-01-01 03:10:00,"20, 25",3,True
333,2304510,2018-01-01 03:20:00,20,1,True
3760,2304510,2018-01-02 05:50:00,25,1,True
3780,2304510,2018-01-02 06:10:00,20,1,True
4022,2304510,2018-01-02 08:00:00,"20, 25",3,True


,StationId,TimeStamp,Alarmcode,alarm_count,alarm_occurred
168,2304510,2018-01-01 01:30:00,NaN,0,False
189,2304510,2018-01-01 01:40:00,NaN,0,False
210,2304510,2018-01-01 01:50:00,"20, 25",6,True
231,2304510,2018-01-01 02:00:00,"20, 25",2,True
252,2304510,2018-01-01 02:10:00,NaN,0,False
